# Apex model: T1 + T2-T1 delta features -> prediction of incident diabetes at T3 (Model C)

Single-model version, combining the up-to-date pipeline from `T1_T2 and T1_T3 model.ipynb` with this notebook, scoped to **Model C only** (`df_trainC.xlsx` / `df_testC.xlsx`) — predicting `DIABETES_T3` from T1 features plus engineered T2-T1 change features. Model A (T1→T2) is intentionally not included here.

**Cohort:** drop missing `HBAC_T1`, exclude prevalent diabetes at baseline (`DIABETES_T1 == 1`) so the task is genuinely incident-case prediction rather than re-detecting existing diabetes, then drop rows missing `DIABETES_T3`. `build_cohort` never reads a `T2` column, so cohort membership itself still never depends on T2 data — T2 only enters as engineered features (see "Delta feature engineering" below), never as an exclusion criterion.

**Features:** every `*_T1` column as-is, including its `*_T1_MISSING` companions and `HBAC_T1` as a continuous predictor, **plus** engineered T2-T1 change features: `{VAR}_DELTA = VAR_T2 - VAR_T1` for continuous T1/T2 pairs (including `HBAC_DELTA`), and `{VAR}_CHANGED` for the one categorical T1/T2 pair (`EDUCATION_LOWER`). Raw `*_T2` values themselves are not kept as separate features — only the delta. This revives the delta-engineering approach from the original conceptual model (`discovery.ipynb`), adapted to Model C's column set. `DIABETES_T1` is explicitly excluded (structurally constant within the cohort by construction), and a zero-variance safety net catches anything else like it. `_MISSING` companion columns aren't always symmetric between train and test (a column exists only where that split had an actual missing value) — reconciled by filling with 0 on whichever side lacks it, rather than dropping the column or crashing. Static/no-suffix baseline columns are excluded by default (`INCLUDE_STATIC_BASELINE_FEATURES`).

**Dtype handling:** each pipeline's preprocessing step is a `ColumnTransformer` built from the actual dtypes of `X` — numeric columns are scaled (logistic regression) or passed through (tree models), any non-numeric column is one-hot encoded automatically. For the tree-based models only, five continuous predictors (`AGE_T1`, `HIP_T1`, `NHDC_T1`, `LDC_T1`, `BRI_T1`) are additionally expanded into B-spline basis features.

**Models:** Logistic Regression + ElasticNet, Random Forest, XGBoost — each wrapped in an `imblearn` pipeline with SMOTE applied only inside the training fold.

**Validation:** one Optuna Bayesian search (TPE sampler + median pruner, 5-fold inner CV) per algorithm, tuned on AUPRC, refit on the full train file, then evaluated on both the train file itself (in-sample, at the same threshold) and the held-out test file. Nothing in tuning or fitting ever sees the test file.

**Threshold:** chosen via inner CV on train data only to maximize F1.

**Metrics:** `% Abnormal` (outcome prevalence), AUROC, AUPRC, precision/recall at the F1-optimal threshold, all with bootstrap confidence intervals — reported for both train and test.

**Interpretation:** SHAP beeswarm plots, a numeric mean-|SHAP| importance table, and (for the best-performing algorithm) a SHAP decision plot — all computed on the test set using the already-fitted final model.

**Also included:** calibration curves + Brier score; a demographics table + figure for the final analytic population; best-algorithm selection (by test AUPRC) with `joblib`/JSON export; and an end-to-end summary bundle (figures + tables + SHAP values) zipped to a single file.

**Data source:** `df_trainC.xlsx`/`df_testC.xlsx` only — no synthetic fallback (see "Load data" below).

## Data provenance (upstream preprocessing)

`df_trainC.xlsx`/`df_testC.xlsx` are produced by a separate team preprocessing pipeline (`Preprocessing.ipynb`), not by this notebook. Conceptual decisions made there, relevant to interpreting the feature set used here:

- **Domain-knowledge column drops:** variables judged not relevant or too redundant for the target analysis (e.g. `WEIGHT_*`, `HEIGHT_*`, `GLU_*`, `HB1C_*` — a duplicate of `HBAC` in different units, `ZIP_CODE`, burnout/quality-of-life subscales) were dropped before any modeling, based on group discussion rather than a fitted statistic.
- **Secondary drops:** additional columns dropped for missingness/garbage data or high correlation with a kept variable (e.g. `WORK_T1`/`WORK_T2`, sleep quality, alcohol/kcal summary scores).
- **Value corrections:** `NSES` thousands-separator parsing fix (`fix_nses`, the same fix used earlier in `discovery.ipynb`); `GENDER` remapped (2->0 female, 1->1 male, confirmed present in the loaded data below); `TYPE2_DIABETES_FAMILY_MOTHER`/`_FATHER` NaN treated as 0 (no).
- **Row-level exclusion:** rows with more than 5 missing values (pre-imputation), across all retained columns, were dropped entirely before any train/test split.
- **Target + HBAC validity:** `DIABETES_{T1,T2,T3} = (HBAC_{T1,T2,T3} > 6.5)`, matching `ensure_diabetes_flags` below. Rows with `HBAC_T1/T2/T3 <= 0` or missing were dropped upstream (garbage/missing lab values), so this notebook's own `HBAC_T1` missingness check is a no-op safety net on the real files, not a live filter.
- **Missingness flags + imputation:** `{col}_MISSING` indicators were added *before* imputation (mode for categorical/binary columns, median for continuous), fit on train and applied to the matching test split — the source of every `*_T1_MISSING` companion column used in this notebook's feature selection.
- **T2 usage is an explicit per-model decision upstream, not a blanket exclusion:** the preprocessing notes flag T2-derived variables as usable for T3-outcome models (Model B/C) but *not* for the T1->T2 model (Model A), since only the former is leakage-free. That's the same reasoning behind reintroducing `*_DELTA`/`*_CHANGED` features in this notebook (see "Delta feature engineering" below) — Model C predicts T3, so T2 information is fair game.

This notebook does not re-run any of the above — it consumes the already-cleaned, already-split, already-imputed `df_trainC.xlsx`/`df_testC.xlsx` as-is.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, SplineTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    precision_recall_curve,
    brier_score_loss,
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

import shap
import joblib

In [ ]:
# --- config -----------------------------------------------------------------
# Tune these down for a fast smoke test, up for a production run.
RANDOM_STATE = 42
DIABETES_THRESHOLD = 6.5

N_INNER_FOLDS = 5
N_OPTUNA_TRIALS = 20
N_BOOTSTRAP = 500

MODEL_NAMES = ["logreg_elasticnet", "random_forest", "xgboost"]

# See the title cell: literal *_T1-only scope was the explicit decision, but
# family history / smoking / NSES are meaningful baseline risk factors excluded
# by that choice. Flip this to True to add them back in.
INCLUDE_STATIC_BASELINE_FEATURES = False

# Model C (T1 -> T3) only - this notebook is intentionally single-model.
TRAIN_PATH_C = "../data/processed/df_trainC.xlsx"
TEST_PATH_C = "../data/processed/df_testC.xlsx"

# All output paths get their own "apex_modelC" namespace, keeping this notebook's exports
# cleanly separated from any other model/notebook writing into the same ../models and
# ../outputs trees.
# Not gitignored automatically - decide separately whether this should be committed.
MODELS_DIR = Path("../models/apex_modelC")
EXPORT_DIR = Path("../outputs/summary_export_modelC")
FIGURES_DIR = EXPORT_DIR / "figures"
TABLES_DIR = EXPORT_DIR / "tables"

# On top of the separate directories above, every individual exported filename is also
# suffixed with OUTPUT_TAG - belt-and-suspenders so a file is unambiguously this
# notebook's output even if it ever gets copied out of its namespaced directory.
OUTPUT_TAG = "apex"

## Load data

`_read_table` dispatches on file extension (`.xlsx`/`.xls` -> `read_excel`, else `read_csv`) — the real files are `.xlsx`, which `read_csv` can't parse (binary format, not text).

Loads `TRAIN_PATH_C`/`TEST_PATH_C` only — `df_trainC.xlsx`/`df_testC.xlsx` are the sole data source for this notebook. There is no synthetic-data fallback: if either file is missing, `_load_model_c_data` raises rather than silently substituting something else.

In [ ]:
import os


def _read_table(path):
    """Dispatch on file extension: Excel files need read_excel, not read_csv (which would try to
    decode the binary .xlsx format as UTF-8 text and raise a UnicodeDecodeError)."""
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    return pd.read_csv(path)


def _load_model_c_data(train_path, test_path):
    """Load the real Model C train/test files. This is the only data source for this notebook -
    no synthetic/raw-CSV fallback - so a missing file is a real problem to surface, not paper over."""
    missing = [p for p in (train_path, test_path) if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(
            f"Model C data file(s) not found: {missing}. This notebook only loads "
            f"{train_path!r} / {test_path!r} - there is no fallback data source."
        )
    return _read_table(train_path), _read_table(test_path)


def ensure_diabetes_flags(df, threshold=DIABETES_THRESHOLD):
    """Derive DIABETES_T1/T2/T3 from HBAC_T1/T2/T3 > threshold if not already present. The real
    preprocessed files already include these as the canonical outcome/exclusion flags (confirmed to
    exactly match HBAC_* > 6.5, but treated as the source of truth in case that definition is ever
    refined upstream, e.g. to also account for diabetes medication)."""
    df = df.copy()
    for t in ["T1", "T2", "T3"]:
        flag_col, hbac_col = f"DIABETES_{t}", f"HBAC_{t}"
        if flag_col not in df.columns and hbac_col in df.columns:
            df[flag_col] = (df[hbac_col] > threshold).astype(int)
    return df


train_c, test_c = _load_model_c_data(TRAIN_PATH_C, TEST_PATH_C)
train_c, test_c = ensure_diabetes_flags(train_c), ensure_diabetes_flags(test_c)

for name, df in {"train_c": train_c, "test_c": test_c}.items():
    print(f"{name}: shape={df.shape}, missing={df.isna().sum().sum()}")

## Delta feature engineering (T2 - T1)

Reintroduces the delta-engineering approach from the original conceptual model (`discovery.ipynb`), adapted to Model C's column set, which — unlike the earlier Model B files — includes `*_T2` versions of several `*_T1` variables. For each continuous variable measured at both visits, add `{VAR}_DELTA = {VAR}_T2 - {VAR}_T1`; for the one categorical T1/T2 pair (`EDUCATION_LOWER`), add a `{VAR}_CHANGED` flag instead of a numeric difference. Raw `*_T2` values themselves (e.g. `HBAC_T2`) are not kept as separate features — only the delta. None of this touches T3, so it introduces no leakage: T2 happens strictly before the T3 outcome being predicted.

In [ ]:
CONTINUOUS_DELTA_BASES = ["AGE", "BKR", "BMI", "BRI", "HBAC", "HBF", "HIP", "LDC", "MAP", "NHDC", "THR"]
CATEGORICAL_CHANGED_BASES = ["EDUCATION_LOWER"]


def engineer_deltas(df, bases=CONTINUOUS_DELTA_BASES):
    """Add {base}_DELTA = {base}_T2 - {base}_T1 for each continuous variable measured at both
    visits. Silently skips any base missing either column."""
    df = df.copy()
    for base in bases:
        t1_col, t2_col = f"{base}_T1", f"{base}_T2"
        if t1_col in df.columns and t2_col in df.columns:
            df[f"{base}_DELTA"] = df[t2_col] - df[t1_col]
    return df


def engineer_changed_flags(df, bases=CATEGORICAL_CHANGED_BASES):
    """Add {base}_CHANGED = 1 if the categorical value differs between T1 and T2, else 0."""
    df = df.copy()
    for base in bases:
        t1_col, t2_col = f"{base}_T1", f"{base}_T2"
        if t1_col in df.columns and t2_col in df.columns:
            df[f"{base}_CHANGED"] = (df[t2_col] != df[t1_col]).astype(int)
    return df


train_c = engineer_changed_flags(engineer_deltas(train_c))
test_c = engineer_changed_flags(engineer_deltas(test_c))

delta_cols = [f"{b}_DELTA" for b in CONTINUOUS_DELTA_BASES if f"{b}_DELTA" in train_c.columns]
changed_cols = [f"{b}_CHANGED" for b in CATEGORICAL_CHANGED_BASES if f"{b}_CHANGED" in train_c.columns]
print(f"Delta features added ({len(delta_cols)}): {delta_cols}")
print(f"Changed-flags added ({len(changed_cols)}): {changed_cols}")

## Cohort construction

Drop missing `HBAC_T1`, exclude prevalent diabetes at baseline (via `DIABETES_T1`), drop rows missing `DIABETES_T3`. `build_cohort` never reads any `T2` column, so cohort membership never depends on T2 data — T2 only enters afterward, as the engineered delta/changed features above.

In [ ]:
def build_cohort(df, target_suffix):
    """Exclusion funnel + binary outcome for a T1-features -> DIABETES_{target_suffix} model.
    target_suffix: 'T2' or 'T3'. Uses the canonical DIABETES_T1/DIABETES_{target_suffix} flags
    (see ensure_diabetes_flags) rather than re-deriving from HBAC_* itself."""
    funnel = {"total": len(df)}

    cohort = df.dropna(subset=["HBAC_T1"])
    funnel["has HBAC_T1"] = len(cohort)

    cohort = cohort[cohort["DIABETES_T1"] == 0]
    funnel["no diabetes at T1"] = len(cohort)

    target_flag_col = f"DIABETES_{target_suffix}"
    cohort = cohort.dropna(subset=[target_flag_col])
    funnel[f"has {target_flag_col}"] = len(cohort)

    cohort = cohort.copy()
    outcome_col = f"{target_flag_col}_OUTCOME"
    cohort[outcome_col] = cohort[target_flag_col].astype(int)
    return cohort, funnel, outcome_col


cohort_c_train, funnel_c_train, outcome_col_c = build_cohort(train_c, "T3")
cohort_c_test, funnel_c_test, _ = build_cohort(test_c, "T3")

for name, (cohort, funnel, outcome_col) in {
    "Model C / train (T1 -> T3)": (cohort_c_train, funnel_c_train, outcome_col_c),
    "Model C / test  (T1 -> T3)": (cohort_c_test, funnel_c_test, outcome_col_c),
}.items():
    print(name)
    for step, n in funnel.items():
        print(f"  {step:20s} n={n:6d}")
    print(f"  prevalence: {cohort[outcome_col].mean():.2%} ({cohort[outcome_col].sum()} / {len(cohort)})")
    print()

## Feature selection

`*_T1` scope (including `*_T1_MISSING` companions), plus the engineered `*_DELTA`/`*_CHANGED` columns and the `HBAC_T2` carve-out from the "Delta feature engineering" step above. `DIABETES_T1` is explicitly excluded: since the cohort filter requires `DIABETES_T1 == 0`, it's *always* 0 within the cohort by construction — a genuinely zero-variance column, which would make `StandardScaler` divide by zero for the logistic regression model. A generic zero-variance safety net (fit on train, applied to test) catches anything else like it.

**Train/test column asymmetry:** a `_MISSING` companion column exists only where that split actually had a missing value, so it can appear in train but not test (or vice versa). `get_shared_t1_features` reconciles this: a `_MISSING` column present on only one side is kept and filled with `0` on the side that lacks it (absence of the column *means* no missingness there); any other one-sided column is dropped from both, since there's no safe default to assume.

In [ ]:
import re

# Columns that are trivially redundant with the cohort exclusion itself, not general-purpose exclusions.
STRUCTURALLY_CONSTANT_COLUMNS = {"DIABETES_T1"}

# Engineered T2-T1 features (see "Delta feature engineering" above) don't match the literal "_T1$"
# pattern below, so they're added to the feature scope explicitly.
DELTA_FEATURE_SUFFIXES = ("_DELTA", "_CHANGED")


def get_t1_features(df, include_static=INCLUDE_STATIC_BASELINE_FEATURES):
    t1_pattern = re.compile(r"_T1(_MISSING)?$")
    t1_cols = [c for c in df.columns if t1_pattern.search(c) and c not in STRUCTURALLY_CONSTANT_COLUMNS]
    engineered_cols = [c for c in df.columns if c.endswith(DELTA_FEATURE_SUFFIXES)]
    cols = t1_cols + engineered_cols
    if include_static:
        static_cols = [
            c
            for c in df.columns
            if not re.search(r"_T[123](_MISSING)?$", c) and c != "ZIP_CODE" and not c.endswith(DELTA_FEATURE_SUFFIXES)
        ]
        cols += static_cols
    return cols


def get_shared_t1_features(train_cohort, test_cohort, include_static=INCLUDE_STATIC_BASELINE_FEATURES):
    """Feature columns from either train or test's T1(+delta) scope, reconciled so both sides can
    actually provide every column. A '*_MISSING' companion present on only one side means that split
    had zero missingness for the underlying variable - kept, and filled with 0 on the side that lacks
    it (see build_model_dataset). Any other column present on only one side is dropped from both,
    since there's no safe default to assume for it."""
    train_cols = set(get_t1_features(train_cohort, include_static))
    test_cols = set(get_t1_features(test_cohort, include_static))

    shared, dropped = [], []
    for c in sorted(train_cols | test_cols):
        if c in train_cols and c in test_cols:
            shared.append(c)
        elif c.endswith("_MISSING"):
            shared.append(c)
        else:
            dropped.append(c)
    if dropped:
        print(f"  dropped (present in only train or only test, no safe default): {dropped}")
    return shared


def build_model_dataset(cohort, outcome_col, feature_cols):
    cohort = cohort.copy()
    for c in feature_cols:
        if c not in cohort.columns:
            cohort[c] = 0  # only '*_MISSING' columns can reach here (see get_shared_t1_features)
    X = cohort[feature_cols].copy()
    y = cohort[outcome_col].copy()
    return X, y


def drop_zero_variance_features(X_train, *other_frames, verbose=True):
    """Drop any feature that's constant in the training data - breaks StandardScaler (div by zero)
    and carries no signal anyway. The decision is made on TRAIN only; the same columns are dropped
    from every frame passed in `other_frames` (e.g. the matching test set) to keep columns aligned."""
    zero_var_cols = X_train.columns[X_train.nunique(dropna=False) <= 1].tolist()
    if zero_var_cols and verbose:
        print(f"  dropping zero-variance features (constant in train): {zero_var_cols}")
    kept_cols = [c for c in X_train.columns if c not in zero_var_cols]
    return (X_train[kept_cols],) + tuple(frame[kept_cols] for frame in other_frames)


print("Model C feature reconciliation:")
feature_cols_c = get_shared_t1_features(cohort_c_train, cohort_c_test)

X_c_train, y_c_train = build_model_dataset(cohort_c_train, outcome_col_c, feature_cols_c)
X_c_test, y_c_test = build_model_dataset(cohort_c_test, outcome_col_c, feature_cols_c)

print("Model C:")
X_c_train, X_c_test = drop_zero_variance_features(X_c_train, X_c_test)

for name, (X, y) in {
    "Model C / train": (X_c_train, y_c_train),
    "Model C / test": (X_c_test, y_c_test),
}.items():
    illegal_t2t3 = [c for c in X.columns if c.endswith(("_T2", "_T3"))]
    assert not illegal_t2t3, f"{name}: must not include raw T2/T3 data: {illegal_t2t3}"
    assert "HBAC_T1" in X.columns, f"{name}: HBAC_T1 must be present as a continuous feature"
    assert "HBAC_DELTA" in X.columns, f"{name}: HBAC_DELTA must be present"
    assert set(y.unique()) <= {0, 1}, f"{name}: outcome must be binary"
    print(f"{name}: X={X.shape}, prevalence={y.mean():.2%}, dtypes={X.dtypes.value_counts().to_dict()}")

print("Feature set = T1 baseline + T2-T1 deltas/changed-flags. No raw T2/T3 data present.")

## Modeling pipeline

Every model is `[preprocessor -> SMOTE -> classifier]`. The preprocessor is a `ColumnTransformer` built from the actual dtypes of whatever `X` it's given: numeric columns are scaled (logistic regression) or passed through as-is (tree models), and any non-numeric column is one-hot encoded (`handle_unknown="ignore"`).

For the tree-based models (random forest, XGBoost) only, `SPLINE_COLUMNS` additionally get expanded into B-spline basis features via `SplineTransformer` — added alongside the raw passthrough values, not replacing them. Not applied to logistic regression. Two of the originally-requested spline variables (triglycerides, HDL cholesterol) no longer exist under those names in the current dataset: triglycerides has no substitute and was dropped; HDL cholesterol is represented via `NHDC_T1` (non-HDL cholesterol, a related but distinct measure), used as the closest available successor.

`imblearn` pipelines automatically skip the resampler at predict/transform time, so SMOTE only ever touches training folds. `k_neighbors` for SMOTE is capped by the minority class count in whatever fold it's fit on.

In [ ]:
def safe_smote(y, random_state=RANDOM_STATE):
    minority_count = pd.Series(y).value_counts().min()
    k_neighbors = max(1, min(5, minority_count - 1))
    return SMOTE(random_state=random_state, k_neighbors=k_neighbors)


SPLINE_COLUMNS = ["AGE_T1", "HIP_T1", "NHDC_T1", "LDC_T1", "BRI_T1"]
SPLINE_DEGREE = 3
SPLINE_N_KNOTS = 5

_missing_spline_cols = [c for c in SPLINE_COLUMNS if c not in X_c_train.columns]
if _missing_spline_cols:
    print(f"Note: spline columns not in current data, will be skipped until available: {_missing_spline_cols}")


def build_preprocessor(X, model_name):
    """ColumnTransformer built from X's actual dtypes: numeric cols get scaled (LR) or passed through
    (tree models), any non-numeric col gets one-hot encoded. For tree models, SPLINE_COLUMNS also get
    a B-spline basis expansion (in addition to the raw passthrough value)."""
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    numeric_step = StandardScaler() if model_name == "logreg_elasticnet" else "passthrough"
    transformers = [("num", numeric_step, numeric_cols)]

    if model_name in ("random_forest", "xgboost"):
        spline_cols = [c for c in SPLINE_COLUMNS if c in numeric_cols]
        if spline_cols:
            transformers.append(
                ("spline", SplineTransformer(degree=SPLINE_DEGREE, n_knots=SPLINE_N_KNOTS), spline_cols)
            )

    if categorical_cols:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols))
    return ColumnTransformer(transformers, remainder="drop")


def make_pipeline(model_name, X_for_columns, y_for_smote, random_state=RANDOM_STATE):
    preprocessor = build_preprocessor(X_for_columns, model_name)
    smote = safe_smote(y_for_smote, random_state)
    if model_name == "logreg_elasticnet":
        clf = LogisticRegression(penalty="elasticnet", solver="saga", max_iter=5000, random_state=random_state)
    elif model_name == "random_forest":
        clf = RandomForestClassifier(random_state=random_state, n_jobs=-1)
    elif model_name == "xgboost":
        clf = XGBClassifier(random_state=random_state, eval_metric="logloss", n_jobs=-1)
    else:
        raise ValueError(f"unknown model_name: {model_name}")
    return ImbPipeline([("preprocessor", preprocessor), ("smote", smote), ("clf", clf)])


def suggest_params(trial, model_name):
    if model_name == "logreg_elasticnet":
        return {
            "clf__C": trial.suggest_float("C", 1e-3, 1e2, log=True),
            "clf__l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        }
    elif model_name == "random_forest":
        return {
            "clf__n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "clf__max_depth": trial.suggest_int("max_depth", 2, 20),
            "clf__min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "clf__max_features": trial.suggest_float("max_features", 0.3, 1.0),
        }
    elif model_name == "xgboost":
        return {
            "clf__n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "clf__max_depth": trial.suggest_int("max_depth", 2, 10),
            "clf__learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "clf__subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "clf__colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "clf__reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        }
    else:
        raise ValueError(f"unknown model_name: {model_name}")

In [ ]:
def best_f1_threshold(y_true, y_proba):
    """Threshold that maximizes F1 on the given predictions."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    if len(thresholds) == 0:
        return 0.5
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-12)
    f1s = f1s[:-1]  # last precision/recall point has no corresponding threshold
    return float(thresholds[np.argmax(f1s)])


def select_threshold(X_train, y_train, model_name, best_params, n_inner_folds=N_INNER_FOLDS, random_state=RANDOM_STATE):
    """F1-optimal threshold from inner-CV out-of-fold predictions on X_train only (never touches any test set)."""
    inner_cv = StratifiedKFold(n_splits=n_inner_folds, shuffle=True, random_state=random_state)
    inner_oof_proba = np.zeros(len(y_train))
    for tr_idx, val_idx in inner_cv.split(X_train, y_train):
        fold_pipe = make_pipeline(model_name, X_train.iloc[tr_idx], y_train.iloc[tr_idx], random_state)
        fold_pipe.set_params(**best_params)
        fold_pipe.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        inner_oof_proba[val_idx] = fold_pipe.predict_proba(X_train.iloc[val_idx])[:, 1]
    return best_f1_threshold(y_train.values, inner_oof_proba)


def make_objective(X_train, y_train, model_name, n_inner_folds=N_INNER_FOLDS, random_state=RANDOM_STATE):
    """Inner-CV objective (mean AUPRC), reporting per-fold progress so Optuna can prune bad trials early."""

    def objective(trial):
        params = suggest_params(trial, model_name)
        inner_cv = StratifiedKFold(n_splits=n_inner_folds, shuffle=True, random_state=random_state)
        fold_scores = []
        for fold_i, (tr_idx, val_idx) in enumerate(inner_cv.split(X_train, y_train)):
            X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

            pipe = make_pipeline(model_name, X_tr, y_tr, random_state)
            pipe.set_params(**params)
            pipe.fit(X_tr, y_tr)
            proba = pipe.predict_proba(X_val)[:, 1]
            fold_scores.append(average_precision_score(y_val, proba))

            trial.report(float(np.mean(fold_scores)), fold_i)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(fold_scores))

    return objective

In [ ]:
def bootstrap_metrics(y_true, y_proba, threshold, n_bootstrap=N_BOOTSTRAP, random_state=RANDOM_STATE):
    """Point estimates + 95% bootstrap CIs for AUROC, AUPRC, precision, recall on pooled predictions."""
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n = len(y_true)

    boot = {"auroc": [], "auprc": [], "precision": [], "recall": []}
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        yt, yp = y_true[idx], y_proba[idx]
        if len(np.unique(yt)) < 2:
            continue
        preds = (yp >= threshold).astype(int)
        boot["auroc"].append(roc_auc_score(yt, yp))
        boot["auprc"].append(average_precision_score(yt, yp))
        boot["precision"].append(precision_score(yt, preds, zero_division=0))
        boot["recall"].append(recall_score(yt, preds, zero_division=0))

    preds = (y_proba >= threshold).astype(int)
    point = {
        "auroc": roc_auc_score(y_true, y_proba),
        "auprc": average_precision_score(y_true, y_proba),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
    }
    ci = {m: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) for m, v in boot.items()}
    return point, ci

## Assemble dataset

`DATASETS` keeps its dict-of-dicts shape (matching `T1_T2 and T1_T3 model.ipynb`) even though there's only one entry here — every downstream function iterates `DATASETS.items()` generically, so nothing else needed to change to drop Model A.

In [ ]:
DATASETS = {
    "T1_to_T3": {"train": (X_c_train, y_c_train), "test": (X_c_test, y_c_test)},
}

## Final model + external test evaluation

For each algorithm: an Optuna search (5-fold inner CV, AUPRC objective, pruned) + refit on the **full train file**, an F1-optimal threshold selected via inner CV on train only, then evaluation on both the train file itself (in-sample, at the same threshold) and the held-out test file. Nothing in the tuning or fitting process ever sees the test file — only the already-fitted final model is used to score it.

In [ ]:
def fit_final_model(X, y, model_name, n_trials=N_OPTUNA_TRIALS, random_state=RANDOM_STATE):
    """Optuna search + refit on the given data. Returns (fitted_pipe, best_params)."""
    sampler = optuna.samplers.TPESampler(seed=random_state)
    pruner = optuna.pruners.MedianPruner(n_warmup_steps=1)
    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
    study.optimize(make_objective(X, y, model_name, random_state=random_state), n_trials=n_trials)

    best_params = {f"clf__{k}": v for k, v in study.best_params.items()}
    pipe = make_pipeline(model_name, X, y, random_state)
    pipe.set_params(**best_params)
    pipe.fit(X, y)
    return pipe, best_params


def evaluate_final_model(X_train, y_train, X_test, y_test, model_name, n_trials=N_OPTUNA_TRIALS, random_state=RANDOM_STATE):
    pipe, best_params = fit_final_model(X_train, y_train, model_name, n_trials=n_trials, random_state=random_state)
    threshold = select_threshold(X_train, y_train, model_name, best_params, random_state=random_state)

    # In-sample (train) predictions: the model scoring the same data it was fit on - optimistic by
    # construction, kept only as a reference point for the train-vs-test comparison below, never as
    # a standalone performance estimate.
    train_proba = pipe.predict_proba(X_train)[:, 1]
    test_proba = pipe.predict_proba(X_test)[:, 1]

    return {
        "model_name": model_name,
        "pipe": pipe,
        "best_params": best_params,
        "y_train_true": y_train.values,
        "y_train_proba": train_proba,
        "y_true": y_test.values,
        "y_proba": test_proba,
        "threshold": threshold,
    }


final_results = {}
for outcome_name, splits in DATASETS.items():
    X_train, y_train = splits["train"]
    X_test, y_test = splits["test"]
    for model_name in MODEL_NAMES:
        print(f"=== [final, external test] {outcome_name} / {model_name} ===")
        final_results[(outcome_name, model_name)] = evaluate_final_model(X_train, y_train, X_test, y_test, model_name)

In [ ]:
def build_performance_summary(results, true_key, proba_key):
    """One row per (outcome, model): % abnormal (positive-class prevalence) + bootstrap metrics at
    the model's selected threshold, evaluated on whichever (true_key, proba_key) pair is passed."""
    rows = []
    for (outcome_name, model_name), res in results.items():
        y_true, y_proba, threshold = res[true_key], res[proba_key], res["threshold"]
        point, ci = bootstrap_metrics(y_true, y_proba, threshold)
        rows.append(
            {
                "outcome": outcome_name,
                "model": model_name,
                "threshold": round(threshold, 3),
                "% Abnormal": f"{100 * np.mean(y_true):.2f}%",
                "AUROC": f"{point['auroc']:.3f} [{ci['auroc'][0]:.3f}, {ci['auroc'][1]:.3f}]",
                "AUPRC": f"{point['auprc']:.3f} [{ci['auprc'][0]:.3f}, {ci['auprc'][1]:.3f}]",
                "Precision": f"{point['precision']:.3f} [{ci['precision'][0]:.3f}, {ci['precision'][1]:.3f}]",
                "Recall": f"{point['recall']:.3f} [{ci['recall'][0]:.3f}, {ci['recall'][1]:.3f}]",
            }
        )
    return pd.DataFrame(rows)


train_summary_df = build_performance_summary(final_results, "y_train_true", "y_train_proba")
test_summary_df = build_performance_summary(final_results, "y_true", "y_proba")

print("Train (in-sample, optimistic) performance at the selected threshold:")
display(train_summary_df)
print()
print("Test (external, unbiased) performance at the selected threshold:")
display(test_summary_df)

## Validation analyses

**Calibration + Brier score** on the final model's predictions on the real test file — the genuine unseen-data check on whether predicted probabilities are trustworthy.

In [ ]:
def plot_calibration(final_results, outcome_name):
    fig, ax = plt.subplots(figsize=(6, 6))
    for (name, model_name), res in final_results.items():
        if name != outcome_name:
            continue
        frac_pos, mean_pred = calibration_curve(res["y_true"], res["y_proba"], n_bins=10, strategy="quantile")
        brier = brier_score_loss(res["y_true"], res["y_proba"])
        ax.plot(mean_pred, frac_pos, marker="o", label=f"{model_name} (Brier={brier:.3f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="perfectly calibrated")
    ax.set_xlabel("mean predicted probability")
    ax.set_ylabel("fraction of positives")
    ax.set_title(f"Calibration on external test - {outcome_name}")
    ax.legend()
    fig.tight_layout()
    plt.show()


for outcome_name in DATASETS:
    plot_calibration(final_results, outcome_name)

## SHAP feature importance

Reuses the final model already fitted on the train file (`final_results[...]["pipe"]`) — no redundant refitting. SHAP values are computed on the **test** file. `TreeExplainer` (RF/XGBoost) or `LinearExplainer` (logistic regression) runs on the preprocessed features; feature names come from `preprocessor.get_feature_names_out()`. Alongside each beeswarm plot, a numeric ranking table (mean absolute SHAP value per feature) is printed and stored in `shap_results`.

In [ ]:
def compute_shap(pipe, X, model_name):
    preprocessor = pipe.named_steps["preprocessor"]
    X_transformed = preprocessor.transform(X)
    # ColumnTransformer prefixes names as "num__COL" / "cat__COL_value"; drop the "num__" prefix for
    # readability, keep "cat__" since one-hot levels need it to stay distinguishable.
    feature_names = [name.removeprefix("num__") for name in preprocessor.get_feature_names_out()]
    X_transformed = pd.DataFrame(X_transformed, columns=feature_names, index=X.index)

    clf = pipe.named_steps["clf"]
    if model_name in ("random_forest", "xgboost"):
        explainer = shap.TreeExplainer(clf)
    else:
        explainer = shap.LinearExplainer(clf, X_transformed)

    shap_values = np.asarray(explainer.shap_values(X_transformed))
    base_value = np.asarray(explainer.expected_value)

    if shap_values.ndim == 3:  # some TreeExplainer versions return (n_samples, n_features, n_classes)
        shap_values = shap_values[:, :, 1]
    base_value = base_value.reshape(-1)
    base_value = float(base_value[1] if base_value.size > 1 else base_value[0])  # positive-class base rate

    return shap_values, X_transformed, base_value


def summarize_shap_importance(shap_values, X_transformed, top_n=15):
    """Mean absolute SHAP value per feature, descending - the numeric counterpart to the beeswarm plot."""
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    importance = pd.Series(mean_abs_shap, index=X_transformed.columns, name="mean_abs_shap")
    return importance.sort_values(ascending=False).head(top_n).to_frame()

In [ ]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

shap_results = {}
for outcome_name, splits in DATASETS.items():
    X_test, _ = splits["test"]
    for model_name in MODEL_NAMES:
        pipe = final_results[(outcome_name, model_name)]["pipe"]
        shap_values, X_transformed, base_value = compute_shap(pipe, X_test, model_name)
        importance_table = summarize_shap_importance(shap_values, X_transformed)
        shap_results[(outcome_name, model_name)] = {
            "shap_values": shap_values,
            "X_transformed": X_transformed,
            "base_value": base_value,
            "importance": importance_table,
        }

        plt.figure()
        shap.summary_plot(shap_values, X_transformed, show=False, max_display=15)
        plt.title(f"{outcome_name} - {model_name} (test set)")
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f"shap_beeswarm_{outcome_name}_{model_name}_{OUTPUT_TAG}.png", dpi=150, bbox_inches="tight")
        plt.show()

        print(f"Top SHAP feature importances (mean |SHAP|) - {outcome_name} / {model_name}:")
        display(importance_table)
        print()

## Demographics table

Baseline (T1) characteristics of Model C's **final analytic population** (train + test combined, after all cohort exclusions). Continuous variables are summarized as mean (SD); binary variables (coded 0/1) as n (%) of the coded-1 group, auto-detected by cardinality (`<= 10` unique values -> binary). `*_MISSING` indicator columns and the structurally-constant `DIABETES_T1` are excluded from the table itself.

In [ ]:
DEMOGRAPHIC_CARDINALITY_THRESHOLD = 10


def is_binary_demographic(series, threshold=DEMOGRAPHIC_CARDINALITY_THRESHOLD):
    return series.nunique(dropna=False) <= threshold


def summarize_demographic_variable(series):
    if is_binary_demographic(series):
        n_positive = int((series == 1).sum())
        pct = 100 * series.mean()
        return f"{n_positive} ({pct:.1f}%)"
    return f"{series.mean():.2f} ({series.std():.2f})"


def get_demographic_variables(df):
    exclude = {"DIABETES_T1"}
    return [
        c for c in df.columns
        if (c.endswith("_T1") or c == "GENDER") and not c.endswith("_MISSING") and c not in exclude
    ]


def build_demographics_table(cohorts):
    """cohorts: {label: dataframe}. One row per baseline variable, one column per cohort."""
    demo_vars = get_demographic_variables(next(iter(cohorts.values())))

    rows = [{"Variable": "N", **{label: len(df) for label, df in cohorts.items()}}]
    for var in demo_vars:
        row = {"Variable": var}
        for label, df in cohorts.items():
            row[label] = summarize_demographic_variable(df[var])
        rows.append(row)

    return pd.DataFrame(rows).set_index("Variable")


cohort_c_full = pd.concat([cohort_c_train, cohort_c_test], ignore_index=True)

DEMOGRAPHIC_COHORTS = {"Model C (T1 -> T3)": cohort_c_full}

demographics_table = build_demographics_table(DEMOGRAPHIC_COHORTS)
print("Continuous: mean (SD). Binary: n (%) of coded 1.")
demographics_table

In [ ]:
def plot_demographics(cohorts, continuous_vars, binary_vars):
    """Boxplots for continuous variables + grouped bar charts for binary variables, comparing cohorts."""
    n_panels = len(continuous_vars) + 2  # +1 for the headline binary var, +1 for the rest grouped
    n_cols = 3
    n_rows = -(-n_panels // n_cols)  # ceil
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows))
    axes = np.atleast_1d(axes).ravel()

    labels = list(cohorts.keys())

    for ax, var in zip(axes, continuous_vars):
        data = [df[var].dropna() for df in cohorts.values()]
        ax.boxplot(data, tick_labels=labels)
        ax.set_title(var)
        ax.tick_params(axis="x", rotation=15)

    headline_var, rest_vars = binary_vars[0], binary_vars[1:]

    ax = axes[len(continuous_vars)]
    pct_positive = [100 * df[headline_var].mean() for df in cohorts.values()]
    ax.bar(labels, pct_positive, color="#4C72B0")
    ax.set_title(f"{headline_var} (% coded 1)")
    ax.set_ylabel("%")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=15)

    ax = axes[len(continuous_vars) + 1]
    x = np.arange(len(rest_vars))
    width = 0.8 / len(cohorts)
    for i, (label, df) in enumerate(cohorts.items()):
        pct = [100 * df[v].mean() for v in rest_vars]
        ax.bar(x + i * width, pct, width, label=label)
    ax.set_xticks(x + width * (len(cohorts) - 1) / 2)
    ax.set_xticklabels(rest_vars, rotation=30, ha="right")
    ax.set_ylabel("%")
    ax.set_title("Other baseline binary characteristics")
    ax.legend(fontsize=8)

    for ax in axes[len(continuous_vars) + 2:]:
        ax.axis("off")

    fig.suptitle("Baseline (T1) characteristics - Model C cohort", fontsize=14)
    fig.tight_layout()
    plt.show()


demo_vars = get_demographic_variables(cohort_c_full)
continuous_vars = [v for v in demo_vars if not is_binary_demographic(cohort_c_full[v])]
binary_vars = [v for v in demo_vars if is_binary_demographic(cohort_c_full[v])]
# GENDER first if present, so it becomes the headline binary panel
binary_vars = sorted(binary_vars, key=lambda v: v != "GENDER")

plot_demographics(DEMOGRAPHIC_COHORTS, continuous_vars, binary_vars)

## Best model selection and export

**Illustrate:** a grouped bar chart comparing all 3 algorithms across AUROC, AUPRC, precision, and recall on the external test set, with 95% bootstrap error bars.

**Select:** the algorithm with the highest test-set AUPRC is chosen as "best" — the same metric used to drive hyperparameter tuning, and more informative than AUROC given the low outcome prevalence.

**Export:** the winning fitted `Pipeline` is saved with `joblib.dump`, plus a sidecar JSON with the decision threshold, tuned hyperparameters, expected feature columns, and test-set performance. Both land in `../models/apex_modelC/` — a dedicated subdirectory, keeping this notebook's exports separated from anything else writing into `../models/`.

**Naming:** every exported filename (here and throughout the rest of the notebook) is additionally suffixed with `OUTPUT_TAG` (`"apex"`, set in the config cell) — e.g. `model_c_best_random_forest_apex.joblib` — so a file is unambiguously this notebook's output even if it's copied out of its namespaced directory.

In [ ]:
def plot_model_comparison(final_results, outcome_name, metrics=("auroc", "auprc", "precision", "recall")):
    algo_results = {model_name: res for (name, model_name), res in final_results.items() if name == outcome_name}

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(metrics))
    width = 0.8 / len(algo_results)

    for i, (model_name, res) in enumerate(algo_results.items()):
        point, ci = bootstrap_metrics(res["y_true"], res["y_proba"], res["threshold"])
        values = [point[m] for m in metrics]
        lower_err = [point[m] - ci[m][0] for m in metrics]
        upper_err = [ci[m][1] - point[m] for m in metrics]
        ax.bar(x + i * width, values, width, yerr=[lower_err, upper_err], capsize=3, label=model_name)

    ax.set_xticks(x + width * (len(algo_results) - 1) / 2)
    ax.set_xticklabels([m.upper() for m in metrics])
    ax.set_ylim(0, 1)
    ax.set_ylabel("score")
    ax.set_title(f"Algorithm comparison on external test - {outcome_name}")
    ax.legend()
    fig.tight_layout()
    plt.show()


for outcome_name in DATASETS:
    plot_model_comparison(final_results, outcome_name)

In [ ]:
BEST_MODEL_METRIC = "auprc"
OUTCOME_TO_MODEL_LABEL = {"T1_to_T3": "model_c"}
# MODELS_DIR and OUTPUT_TAG are set in the config cell (../models/apex_modelC, "apex").


def select_best_model(final_results, outcome_name, metric=BEST_MODEL_METRIC):
    candidates = {model_name: res for (name, model_name), res in final_results.items() if name == outcome_name}
    scores = {}
    for model_name, res in candidates.items():
        point, _ = bootstrap_metrics(res["y_true"], res["y_proba"], res["threshold"])
        scores[model_name] = point[metric]
    best_model_name = max(scores, key=scores.get)
    return best_model_name, scores


def export_best_model(outcome_name, model_name, feature_columns):
    res = final_results[(outcome_name, model_name)]
    label = OUTCOME_TO_MODEL_LABEL[outcome_name]

    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    pipe_path = MODELS_DIR / f"{label}_best_{model_name}_{OUTPUT_TAG}.joblib"
    meta_path = MODELS_DIR / f"{label}_best_{model_name}_{OUTPUT_TAG}_metadata.json"

    joblib.dump(res["pipe"], pipe_path)

    point, ci = bootstrap_metrics(res["y_true"], res["y_proba"], res["threshold"])
    metadata = {
        "outcome": outcome_name,
        "algorithm": model_name,
        "threshold": res["threshold"],
        "best_params": res["best_params"],
        "feature_columns": list(feature_columns),
        "test_performance": {
            "point": point,
            "ci_95": {k: list(v) for k, v in ci.items()},
        },
    }
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2, default=str)

    print(f"exported: {pipe_path} , {meta_path}")
    return pipe_path, meta_path


best_models = {}
exported_paths = {}
for outcome_name in DATASETS:
    best_name, scores = select_best_model(final_results, outcome_name)
    best_models[outcome_name] = best_name
    print(f"{outcome_name}: best algorithm by {BEST_MODEL_METRIC.upper()} = {best_name}")
    print(f"  scores: {{ {', '.join(f'{k}: {v:.3f}' for k, v in scores.items())} }}")

    feature_columns = DATASETS[outcome_name]["train"][0].columns
    exported_paths[outcome_name] = export_best_model(outcome_name, best_name, feature_columns)

## SHAP decision plot for the best model

Traces individual prediction paths — how each feature's contribution accumulates from the population base rate (`base_value`) up to the final predicted probability, one line per observation. Reuses the already-computed `shap_results` for the winning algorithm (from `best_models` above) — no recomputation. Limited to a random subset of `DECISION_PLOT_N_SAMPLES` test-set observations (fixed seed) to stay legible.

In [ ]:
DECISION_PLOT_N_SAMPLES = 50


def plot_shap_decision(outcome_name, model_name, n_samples=DECISION_PLOT_N_SAMPLES, random_state=RANDOM_STATE):
    res = shap_results[(outcome_name, model_name)]
    shap_values, X_transformed, base_value = res["shap_values"], res["X_transformed"], res["base_value"]

    rng = np.random.default_rng(random_state)
    n = len(X_transformed)
    idx = rng.choice(n, size=min(n_samples, n), replace=False)

    plt.figure(figsize=(9, 8))
    shap.decision_plot(
        base_value,
        shap_values[idx],
        X_transformed.iloc[idx],
        feature_display_range=slice(-1, -16, -1),  # top 15 features, same cap as the beeswarm
        show=False,
    )
    plt.title(f"{outcome_name} - best model ({model_name}): SHAP decision plot (n={len(idx)} test cases)")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"shap_decision_{outcome_name}_{model_name}_{OUTPUT_TAG}.png", dpi=150, bbox_inches="tight")
    plt.show()


for outcome_name, model_name in best_models.items():
    plot_shap_decision(outcome_name, model_name)

## Export summary bundle

The SHAP beeswarm and decision plot figures already saved to `FIGURES_DIR`, the train/test performance tables (`% Abnormal`, AUROC, AUPRC, precision, recall with bootstrap CIs), and the SHAP values themselves — both the raw per-observation array and the mean-|SHAP| importance ranking, for all 3 algorithms. Bundled into `EXPORT_DIR` and zipped to a single file.

`EXPORT_DIR`/`MODELS_DIR` are both namespaced to `..._modelC` (see the config cell), and every individual filename inside is further suffixed with `OUTPUT_TAG` (`"apex"`), so nothing here collides with any other notebook's output even if run against the same data folder. The zip itself is named `summary_export_modelC_apex.zip`. Not gitignored automatically — decide separately whether `../outputs/` and `../models/` should be committed or excluded.

In [ ]:
import shutil

TABLES_DIR.mkdir(parents=True, exist_ok=True)

train_summary_df.to_csv(TABLES_DIR / f"train_performance_summary_{OUTPUT_TAG}.csv", index=False)
test_summary_df.to_csv(TABLES_DIR / f"test_performance_summary_{OUTPUT_TAG}.csv", index=False)

for (outcome_name, model_name), res in shap_results.items():
    shap_df = pd.DataFrame(res["shap_values"], columns=res["X_transformed"].columns)
    shap_df.to_csv(TABLES_DIR / f"shap_values_{outcome_name}_{model_name}_{OUTPUT_TAG}.csv", index=False)
    res["importance"].to_csv(TABLES_DIR / f"shap_importance_{outcome_name}_{model_name}_{OUTPUT_TAG}.csv")

zip_path = shutil.make_archive(str(EXPORT_DIR.parent / f"{EXPORT_DIR.name}_{OUTPUT_TAG}"), "zip", root_dir=EXPORT_DIR)

print(f"exported bundle: {zip_path}")
print(f"  figures: {len(list(FIGURES_DIR.glob('*.png')))} PNGs")
print(f"  tables:  {len(list(TABLES_DIR.glob('*.csv')))} CSVs")